<a href="https://colab.research.google.com/github/JuanGL2715/lstm-text-generation/blob/main/lstm_text_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt
from keras.callbacks import LambdaCallback
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import LSTM
import sys
import random
import io
from keras.layers import Embedding

import requests

# URL del libro Don Quijote de la Mancha del Proyecto Gutenberg
url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"

# Descargar el contenido del libro
texto = requests.get(url).text

# Guardar el texto en un archivo local
with open("don_quijote.txt", "w", encoding="utf-8") as f:
    f.write(texto)

# Ruta al archivo del texto
path = "don_quijote.txt"

In [ ]:
with open(path, encoding="utf8") as f:
    text = f.read().lower()
    # Reemplazar saltos de línea con espacios para que el texto sea una sola línea
    text = text.replace('\n', ' ').replace('\r', ' ')
    # Reemplazar múltiples espacios con un solo espacio para limpiar el texto
    text = ' '.join(text.split())

# Imprimir la longitud total del texto preprocesado
print("Longitud:", len(text))
# Imprimir los primeros 1000 caracteres del texto para verificar el preprocesamiento
print(text[:1000])

Longitud: 2123953
the project gutenberg ebook of don quijote this ebook is for the use of anyone anywhere in the united states and most other parts of the world at no cost and with almost no restrictions whatsoever. you may copy it, give it away or re-use it under the terms of the project gutenberg license included with this ebook or online at www.gutenberg.org. if you are not located in the united states, you will have to check the laws of the country where you are located before using this ebook. title: don quijote author: miguel de cervantes saavedra release date: december 1, 1999 [ebook #2000] most recently updated: january 17, 2021 language: spanish other information and formats: www.gutenberg.org/ebooks/2000 credits: an anonymous project gutenberg volunteer and joaquin cuenca abela *** start of the project gutenberg ebook don quijote *** el ingenioso hidalgo don quijote de la mancha por miguel de cervantes saavedra el ingenioso hidalgo don quijote de la mancha tasa testimonio de 

In [ ]:
# Encontrar el índice de inicio del texto relevante del Quijote
inicio = text.find("el ingenioso hidalgo don quijote de la mancha")

# Recortar el texto para que comience desde esa frase
text = text[inicio:]

In [ ]:
# Esta celda estaba vacía y se puede usar para depuración, por ejemplo, para ver la longitud del texto.
# len(text)

In [ ]:
# Imprimir la longitud actualizada del texto
print("Longitud del texto: {}".format(len(text)))
# Imprimir los primeros 500 caracteres del texto recortado
print(text[0:500])

Longitud del texto: 2123115
el ingenioso hidalgo don quijote de la mancha por miguel de cervantes saavedra el ingenioso hidalgo don quijote de la mancha tasa testimonio de las erratas el rey al duque de béjar prólogo al libro de don quijote de la mancha que trata de la condición y ejercicio del famoso hidalgo don quijote de la mancha que trata de la primera salida que de su tierra hizo el ingenioso don quijote donde se cuenta la graciosa manera que tuvo don quijote en armarse caballero de lo que le sucedió a nuestro caball


In [ ]:
# Obtener todos los caracteres únicos en el texto y ordenarlos
chars = sorted(list(set(text)))

# Crear un diccionario para mapear cada carácter a un índice entero
char_indices = dict((c, i) for i, c in enumerate(chars))

# Crear un diccionario inverso para mapear cada índice entero a un carácter
indice_char = dict((i, c) for i, c in enumerate(chars))

In [ ]:
# Mostrar el diccionario de mapeo de caracteres a índices
char_indices
# Mostrar el número total de caracteres únicos
# len(chars) # se obtienen 76 caracteres únicos

{' ': 0,
 '!': 1,
 '"': 2,
 '#': 3,
 '$': 4,
 '%': 5,
 "'": 6,
 '(': 7,
 ')': 8,
 '*': 9,
 '+': 10,
 ',': 11,
 '-': 12,
 '.': 13,
 '/': 14,
 '0': 15,
 '1': 16,
 '2': 17,
 '3': 18,
 '4': 19,
 '5': 20,
 '6': 21,
 '7': 22,
 '8': 23,
 '9': 24,
 ':': 25,
 ';': 26,
 '?': 27,
 ']': 28,
 'a': 29,
 'b': 30,
 'c': 31,
 'd': 32,
 'e': 33,
 'f': 34,
 'g': 35,
 'h': 36,
 'i': 37,
 'j': 38,
 'k': 39,
 'l': 40,
 'm': 41,
 'n': 42,
 'o': 43,
 'p': 44,
 'q': 45,
 'r': 46,
 's': 47,
 't': 48,
 'u': 49,
 'v': 50,
 'w': 51,
 'x': 52,
 'y': 53,
 'z': 54,
 '¡': 55,
 '«': 56,
 '»': 57,
 '¿': 58,
 'à': 59,
 'á': 60,
 'é': 61,
 'í': 62,
 'ï': 63,
 'ñ': 64,
 'ó': 65,
 'ù': 66,
 'ú': 67,
 'ü': 68,
 '—': 69,
 '‘': 70,
 '’': 71,
 '“': 72,
 '”': 73,
 '•': 74,
 '™': 75}

In [ ]:
# Definir la longitud de las secuencias de entrada para el modelo
SEQ_LENGTH = 35
# Definir el paso (step) para tomar las secuencias, es decir, cada cuántos caracteres se inicia una nueva secuencia
step = 3
# Listas para almacenar las secuencias de entrada (rawX) y los caracteres objetivo (rawy)
rawX = []
rawy = []

# Iterar sobre el texto para crear las secuencias de entrada y los caracteres objetivo
for i in range(0, len(text) - SEQ_LENGTH, step):
    rawX.append(text[i: i+SEQ_LENGTH]) # Secuencia de entrada de SEQ_LENGTH caracteres
    rawy.append(text[i+SEQ_LENGTH])    # Carácter siguiente a la secuencia

In [ ]:
# Calcular el número total de secuencias generadas
n_sentences = len(rawX)

# Imprimir el número de secuencias
print(n_sentences)

707694


In [ ]:
# Establecer un límite máximo para el número de secuencias a utilizar en el entrenamiento
MAX_SEQUENCES = 50000

# Permutar aleatoriamente las secuencias y sus caracteres objetivo
perm = np.random.permutation(len(rawX)) #Permutar aleatoriamente una secuencia, o devolver un rango permutado.
rawX, rawy = np.array(rawX), np.array(rawy)
rawX, rawy = rawX[perm], rawy[perm]
# Tomar solo el número máximo de secuencias definido
rawX, rawy = list(rawX[:MAX_SEQUENCES]), list(rawy[:MAX_SEQUENCES])

# Imprimir la longitud final del conjunto de datos de entrenamiento
print(len(rawX))

50000


In [ ]:
# Inicializar arrays numpy para las entradas (X) y salidas (y) del modelo
# X será un array 2D donde cada fila es una secuencia de caracteres representada por sus índices
X = np.zeros((len(rawX), SEQ_LENGTH), dtype=np.int32)
# y será un array 1D donde cada elemento es el índice del carácter objetivo para cada secuencia
y = np.zeros((len(rawX),), dtype=np.int32)

# Convertir las secuencias de caracteres a secuencias de índices enteros
for i, sentence in enumerate(rawX):
    for t, char in enumerate(sentence):
        X[i, t] = char_indices[char] # Mapear cada carácter al índice correspondiente
    y[i] = char_indices[rawy[i]] # Mapear el carácter objetivo al índice correspondiente

In [ ]:
# Definir la arquitectura del modelo Keras
model = Sequential([
    # Capa de Embedding: convierte índices enteros en vectores densos de tamaño output_dim
    # input_dim es el tamaño del vocabulario (número de caracteres únicos)
    # output_dim es la dimensión de los vectores de embedding
    Embedding(input_dim=len(chars), output_dim=64),
    # Capa LSTM: capa de memoria a largo plazo para procesar secuencias
    LSTM(128),
    # Capa Densa (fully connected): capa de salida que mapea las representaciones LSTM a las probabilidades de los caracteres
    # len(chars) es el número de caracteres posibles (clases)
    # activation='softmax' asegura que la salida sea una distribución de probabilidad sobre los caracteres
    Dense(len(chars), activation='softmax')
])

# Compilar el modelo
model.compile(
    optimizer='adam', # Optimizador Adam para ajustar los pesos del modelo
    loss='sparse_categorical_crossentropy', # Función de pérdida adecuada para etiquetas enteras (no one-hot encoded)
    metrics=['accuracy'] # Métrica para evaluar el rendimiento del modelo durante el entrenamiento
)

In [ ]:
# Esta celda contiene código comentado para el entrenamiento del modelo, que se ha movido a la celda 3oT7pNvjrP2e.
# model.fit(X, y, batch_size=128, epochs=20, verbose=0)

In [ ]:
def sample(probs, temperature=1.0):
    """Nos da el índice del elemento a elegir según la distribución
    de probabilidad dada por probs.

    Args:
      probs es la salida dada por una capa softmax:
        probs = model.predict(x_to_predict)[0]

      temperature es un parámetro que nos permite obtener mayor
        "diversidad" a la hora de obtener resultados.

        temperature = 1 nos da la distribución normal de softmax
        0 < temperature < 1 hace que el sampling sea más conservador,
          de modo que sampleamos cosas de las que estamos más seguros
        temperature > 1 hace que los samplings sean más atrevidos,
          eligiendo en más ocasiones clases con baja probabilidad.
          Con esto, tenemos mayor diversidad pero se cometen más
          errores.
    """
    # Cast a float64 por motivos numéricos para evitar problemas de precisión
    probs = np.asarray(probs).astype('float64')

    # Aplicar logaritmo de probabilidades y reducción por temperatura.
    # Esto suaviza o agudiza la distribución de probabilidad
    probs = np.log(probs) / temperature

    # Volver a aplicar la función exponencial y normalizar de nuevo
    # para obtener una distribución de probabilidad válida
    exp_probs = np.exp(probs)
    probs = exp_probs / np.sum(exp_probs)

    # Realizar el sampling de acuerdo a las nuevas probabilidades
    # np.random.multinomial elige un índice basado en la distribución de probabilidades
    samples = np.random.multinomial(1, probs, 1)
    # Devolver el índice del carácter seleccionado
    return np.argmax(samples)

In [ ]:
TEMPERATURES_TO_TRY = [0.2, 0.5, 1.0, 1.2] # Temperaturas a probar para la generación de texto
GENERATED_TEXT_LENGTH = 300 # Longitud del texto a generar

def generate_text(seed_text, model, length=300, temperature=1, max_length=30):
    """Genera una secuencia de texto a partir de seed_text utilizando model.

    La secuencia tiene longitud length y el sampling se hace con la temperature
    definida.
    """

    # Aquí guardaremos nuestro texto generado, que incluirá el
    # texto origen
    generated = seed_text

    # Utilizar el modelo en un bucle para generar carácter a carácter.
    # Se construye x_pred de manera similar a como se hizo en el preprocesamiento,
    # pero solo para una secuencia. La secuencia de entrada al
    # modelo debe actualizarse con el último caracter predicho
    prediction = []

    for i in range(length):
        # Crear un array numpy para la secuencia de entrada del modelo
        # X para predicción debe ser un array 2D de índices de caracteres (batch_size=1, SEQ_LENGTH)
        x_pred = np.zeros((1, SEQ_LENGTH), dtype=np.int32)

        # Rellenar con los índices de los caracteres de la secuencia semilla actual
        for t, char in enumerate(seed_text):
            x_pred[0, t] = char_indices[char]

        # Generar la predicción para el siguiente carácter
        # Usar la llamada directa del modelo para inferencia sin rastreo de tf.function
        preds = model(x_pred, training=False)[0]
        # Elegir un carácter de las probabilidades de predicción usando la función sample
        next_index = sample(preds, temperature)
        next_char = indice_char[next_index]

        prediction.append(next_char)
        # Añadir el carácter predicho a la secuencia semilla para que la próxima predicción
        # incluya este carácter. Se "desliza" la ventana de la secuencia.
        # generated += next_char # Esto acumularía el texto generado desde el inicio
        seed_text = seed_text[1:] + next_char # Actualizar la semilla para la siguiente predicción

        print(next_char, end= " "); # Imprimir el carácter generado
        # sys.stdout.flush() # Vaciar el buffer de salida para ver la predicción en tiempo real

    prediction = ''.join(prediction) # Unir los caracteres predichos para formar el texto generado
    sys.stdout.flush() # Vaciar el buffer final

    ### FIN DE TU CÓDIGO
    return generated # Retornar el texto generado (que aquí es la semilla original)

In [ ]:
import tensorflow as tf
from keras.models import Sequential # Importar Sequential de keras.models
from keras.layers import Embedding, LSTM, Dense # Importar las capas necesarias

tf.config.run_functions_eagerly(True) # Habilitar la ejecución eager de TensorFlow para depuración

# Redefinir y compilar el modelo para asegurar la configuración correcta de la forma de entrada
model = Sequential([
    # Capa de Embedding: convierte índices enteros en vectores densos.
    # input_dim: tamaño del vocabulario (número de caracteres únicos).
    # output_dim: dimensión de los vectores de embedding (64 en este caso).
    # input_length: longitud de las secuencias de entrada (SEQ_LENGTH).
    Embedding(input_dim=len(chars), output_dim=64, input_length=SEQ_LENGTH),
    # Capa LSTM: una capa recurrente con 128 unidades.
    LSTM(128),
    # Capa Densa: capa de salida con un número de neuronas igual al tamaño del vocabulario.
    # activation='softmax' para obtener probabilidades sobre todos los caracteres posibles.
    Dense(len(chars), activation='softmax')
])

# Compilar el modelo
model.compile(
    optimizer='adam', # Optimizador Adam, popular por su eficiencia
    loss='sparse_categorical_crossentropy', # Función de pérdida para clasificación multiclase cuando las etiquetas son enteros
    metrics=['accuracy'] # Métrica de precisión para monitorear el rendimiento
)

# Entrenar el modelo
model.fit(
    X, # Datos de entrada (secuencias de índices de caracteres)
    y, # Etiquetas (carácter siguiente a cada secuencia)
    batch_size=128, # Tamaño del lote para el entrenamiento
    epochs=10, # Número de épocas de entrenamiento
    callbacks=[generation_callback] # Callback para generar texto al final de cada época
)

Epoch 1/10
  1/391 ━━━━━━━━━━━━━━━━━━━━ 53s 137ms/step - accuracy: 0.0234 - loss: 4.3296

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.2131 - loss: 2.9409



------> Epoch: 1 - Generando texto con temperature 0.2
d e   l o   d e   t e   q u e   c o r a   d e   c a r a   l o   d e   d e   d e   d e   d e   q u e   d e   d e   d a   d e   c a r   e n   s e   q u e   d e   l a   m a n   s e   a l   q u e   d a r a   d e   s e   l o   d e   q u e   l a   c e r   e   c u e   d e r   e n   d e   d e   q u e   l a   c a s   e n   d i r o   d o   d e   p e n   e n   e n   d e   d e   d e   c o r   d e   s e   p u e   c u e   d e   q u e   l e   m e   t a s   d e   q u e   y   e n   e n   q u e   q u e   q u e   h a   q u e   s o   d e   d e   d e   q u e   s e   l a   l a   q u e   a l   d e   m a s   s e   l Seed: eligro de quitármela para volverla 
Texto generado: eligro de quitármela para volverla 
------> Epoch: 1 - Generando texto con temperature 0.5
d e n t a   e n   d e   p o r a   d e n   e n   q u e   e   a s   p i e   n e   c o n   d a n   p i s a s   a l a r  

## Análisis del Texto Generado en la Época 10

Después de 10 épocas de entrenamiento, podemos observar mejoras en la coherencia del texto generado, aunque la naturaleza de un modelo a nivel de caracteres sigue presentando desafíos.

**1. Temperatura: 0.5 (Moderado)**

**Análisis:** Con esta temperatura, el modelo sigue mostrando un patrón de espaciado entre caracteres, lo que dificulta la lectura. Sin embargo, se pueden identificar más fragmentos de palabras y pseudopalabras que parecen estar aprendiendo de las estructuras del español (`señor despenta`, `cantares`, `cuanto`, `migos`). La repetición ha disminuido ligeramente en comparación con la Época 1, pero aún predomina la formación de palabras segmentadas o mal formadas.

**2. Temperatura: 1.0 (Más Creativo)**

   **Análisis:** A esta temperatura, el modelo intenta generar un texto más diverso y menos predecible. Se ven más caracteres únicos y combinaciones que se asemejan a palabras (ej., `piela`, `sus`, `muncer`, `sacado`). Aunque la coherencia sintáctica y semántica aún es débil, hay una clara tendencia a la variación. La puntuación como el punto y coma se utiliza, lo que sugiere un aprendizaje de las estructuras del texto de entrenamiento. Sin embargo, todavía hay muchas palabras inventadas o mal escritas que hacen que el texto sea incomprensible.

**3. Temperatura: 1.2 (Experimental)**

   **Análisis:** A la temperatura más alta, el modelo es el más aventurero, introduciendo una mayor diversidad de caracteres y estructuras. Se observan palabras más inusuales y a veces combinaciones de caracteres que no se encuentran en el idioma español, como `arragha`, `atoges`, `smpovey`. La inclusión de comillas `"` y guiones `—` muestra que el modelo ha capturado algunas características del texto original, pero las aplica de forma caótica. El resultado es un texto muy variado pero casi completamente incomprensible, lo que es típico de temperaturas altas donde la aleatoriedad es priorizada sobre la plausibilidad lingüística.

**Conclusión General**
El modelo ha progresado en aprender algunas distribuciones de caracteres y la estructura básica de las palabras, pero sigue luchando con la coherencia a nivel de frase o párrafo. Las temperaturas más bajas producen texto repetitivo pero más "seguro", mientras que las temperaturas más altas generan texto más "creativo" pero a menudo sin sentido. Para generar texto más legible y coherente, se requerirían más épocas de entrenamiento y, posiblemente, ajustes en la arquitectura del modelo o en la forma de preprocesar los datos.